## 闭包
rust闭包的默认捕获策略是：尽可能使用借用，只有在不得不move的时候才move。 优先级如下：
1. 尝试不可变借用(&T) -- 对应Fn
2. 尝试可变借用(&mut T) -- 对应FnMut
3. 尝试move -- 对应FnOnce， 需要使用move关键字
### 结构体中的闭包
如下是一个极简缓存结构体。
```rust
struct Cacher<T>
where
    T:Fn(u32)->u32
    {
        query:T,
        value:Option<u32>,
    }
```
***Fn特征不仅仅适用于闭包，还适用于函数。 上面的query字段除了可以使用闭包作为值外，还可以使用一个函数来作为值***

当闭包从环境中捕获一个值时， 会分配内存去存储这些值，在某些场景，内存分配时一个额外负担。

### 三种Fn特征
1. FnOnce  所有权转移， 只能调用一次
2. FnMut   可变引用， 可以调用多次
3. Fn      不可变引用， 可以调用多次

**FnOnce**:
1. 该类型的闭包只能调用一次。
2. 可以对FnOnce类型的闭包添加Copy特征，此时就可以二次调用了。
3. <font color=red>我想要强制拥有捕获变量的的所有权，可以使用move关键字。</font>
```rust
use std::thread;
let v = vec![1,2,3];
let handle = thread::spawn (move || {
    println!("{:?}", v);
});
handle.join().unwrap();
`````
**FnMut**
```rust
let mut s = String::new();
let mut update_string = |str| s.push_str(str);
update_string("hello");
println!("{:?}", s);
```
或者：
```rust
let s = String::new();
let update_string = |str| s.push_str(str);
exec(update_string);
println!("{:?}", S);


Fn exec<'a, F: FnMut(&'a str)>(mut f: F) {
    f("hello")
}
```
上述代码能正常编译执行。 update_string是一个不可变类型， 而exec的参数接收一个可变类型。 类型不匹配，<font color=red>为什么没有报错呢？</font>
为什么闭包变量没有用mut修饰却能赋值给一个可变类型的闭包？   事实上， FnMut只是trait的名字，声明为FnMut和要不要mut没什么关系，**FnMut是推导出的特征类型**。mut是rust语言层面的一个修饰符，用于声明一个绑定是可变的。
***exec的参数是一个特征 而不是一个变量。***
关于闭包的copy特征， 只要闭包捕获的类型都实现了copy特征的话，这个闭包类型会默认实现copy特征。 否则需要显示声明特征实现Copy特征

***一个闭包实现了那种Fn特征取决于该闭包如何使用被捕获的变量， 而不是取决于闭包如何捕获他们***
#### 三种Fn的关系
* 所有闭包都自动实现了FnOnce特征， 因此任何一个闭包的都至少可以调用一次
* 没有移出捕获变量的所有权的闭包自动实现了FnMut特征
* 不需要对捕获变量进行修改的闭包自动实现了Fn特征

<font color=red>Rust要求函数的参数和返回类型必须有骨钉的内存大小</font>。
<font color=red>就算签名一样的闭包，类型也是不同的</font>

## 迭代器
### For循环与迭代器
实现IntoIterator特征， rust可以通过for语法糖，自动将目标类型转换为迭代器
例如数组实现了IntoIterator特征， 因此我们可以通过for去迭代访问里面的元素

#### IntoIterator特征及相关方法 

### 惰性初始化





 